# AI RMF report demo

This notebook loads a `BenchmarkResult` produced by notebook 02, feeds it to `AIRMFReporter`, and renders both markdown and HTML. It ends with a short note on how each section of the output corresponds to the NIST AI RMF Govern / Map / Measure / Manage functions.

## 1. Load a saved BenchmarkResult

The JSON was written by notebook 02. If you don't have it yet, run notebook 02 first or substitute any other `BenchmarkResult.model_dump_json()` output.

In [ ]:
from pathlib import Path

from lub.types import BenchmarkResult

raw = Path("finqa_result.json").read_text(encoding="utf-8")
result = BenchmarkResult.model_validate_json(raw)
result

## 2. Render markdown and HTML

`AIRMFReporter` takes a list of results because a real report typically aggregates several runs (different estimators, different datasets, different seeds). For this demo we wrap a single result in a list.

In [ ]:
from lub.reports import AIRMFReporter

reporter = AIRMFReporter(results=[result], title="FinQA smoke test")
md = reporter.render("md")
html = reporter.render("html")
Path("airmf_report.md").write_text(md, encoding="utf-8")
Path("airmf_report.html").write_text(html, encoding="utf-8")

## 3. Inline preview

In [ ]:
from IPython.display import HTML, Markdown

Markdown(md)

In [ ]:
HTML(html)

## 4. How this maps to the AI RMF

- **Govern** — the report header documents *who* ran *what*, under which git SHA and dependency fingerprint. That is the accountability trail a second-line reviewer needs.
- **Map** — the narrative block declares intended use (QA in regulated banking workflows) and explicit out-of-scope uses (training, RAG, agentic tool use). Scope boundaries are the whole point of the Map function.
- **Measure** — the per-run table shows accuracy, ECE, refusal AUROC, and the dataset hash, each annotated with its RMF sub-category (MEASURE 2.3 / 2.7 / 2.8 / 2.9). This is the evidence block.
- **Manage** — the closing section restates the refusal policy and change-management rules. Any drift in those needs a fresh report, not an in-place edit.

A reviewer reads the report top-to-bottom and should be able to answer, in order: *who is accountable, what is in scope, does the model work, and how is it being run*. If any one of those answers is missing, the report has a gap.